<a href="https://colab.research.google.com/github/mohamedalaaaz/testpytroch/blob/main/al%20web%20defend%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load data
data = pd.read_csv('web_traffic.csv')

# Features and labels
X = data.drop('is_bot', axis=1).values
y = data['is_bot'].values

# Normalize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

# Create Dataset and DataLoader
class TrafficDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = TrafficDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Define a simple neural network
class BotDetector(nn.Module):
    def __init__(self, input_dim):
        super(BotDetector, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

model = BotDetector(input_dim=X.shape[1])

# Loss and optimizer
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training loop
for epoch in range(20):
    model.train()
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

    print(f"Epoch [{epoch+1}/20], Loss: {loss.item():.4f}")

# Evaluate
model.eval()
with torch.no_grad():
    predictions = model(X_test_tensor)
    predicted_classes = (predictions > 0.5).float()
    accuracy = (predicted_classes == y_test_tensor).float().mean()
    print(f"Test Accuracy: {accuracy:.4f}")


In [2]:
# generate_traffic_data.py

import pandas as pd
import numpy as np

np.random.seed(42)

def generate_data(num_samples=1000):
    data = []

    for _ in range(num_samples):
        # Randomly decide if this is a bot or not
        is_bot = np.random.rand() < 0.3  # 30% bots

        if is_bot:
            request_rate = np.random.uniform(3, 10)
            user_agent_score = np.random.uniform(0.0, 0.4)
            time_on_page = np.random.uniform(1, 10)
            navigation_depth = np.random.randint(1, 3)
        else:
            request_rate = np.random.uniform(0.1, 2)
            user_agent_score = np.random.uniform(0.6, 1.0)
            time_on_page = np.random.uniform(30, 300)
            navigation_depth = np.random.randint(2, 10)

        data.append([request_rate, user_agent_score, time_on_page, navigation_depth, int(is_bot)])

    df = pd.DataFrame(data, columns=['request_rate', 'user_agent_score', 'time_on_page', 'navigation_depth', 'is_bot'])
    df.to_csv('web_traffic.csv', index=False)
    print("✅ Generated web_traffic.csv with", num_samples, "samples.")

generate_data(2000)


✅ Generated web_traffic.csv with 2000 samples.


In [3]:
# train_bot_detector.py

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load data
data = pd.read_csv('web_traffic.csv')
X = data.drop('is_bot', axis=1).values
y = data['is_bot'].values

# Normalize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

class TrafficDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = TrafficDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

class BotDetector(nn.Module):
    def __init__(self, input_dim):
        super(BotDetector, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

model = BotDetector(input_dim=X.shape[1])

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(20):
    model.train()
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
    print(f"Epoch [{epoch+1}/20], Loss: {loss.item():.4f}")

model.eval()
with torch.no_grad():
    predictions = model(X_test_tensor)
    predicted_classes = (predictions > 0.5).float()
    accuracy = (predicted_classes == y_test_tensor).float().mean()
    print(f"✅ Test Accuracy: {accuracy:.4f}")

# Save model and scaler
torch.save(model.state_dict(), 'bot_model.pth')
import joblib
joblib.dump(scaler, 'scaler.pkl')


Epoch [1/20], Loss: 0.5659
Epoch [2/20], Loss: 0.4080
Epoch [3/20], Loss: 0.2823
Epoch [4/20], Loss: 0.1114
Epoch [5/20], Loss: 0.0818
Epoch [6/20], Loss: 0.0471
Epoch [7/20], Loss: 0.0220
Epoch [8/20], Loss: 0.0187
Epoch [9/20], Loss: 0.0092
Epoch [10/20], Loss: 0.0108
Epoch [11/20], Loss: 0.0052
Epoch [12/20], Loss: 0.0070
Epoch [13/20], Loss: 0.0026
Epoch [14/20], Loss: 0.0028
Epoch [15/20], Loss: 0.0051
Epoch [16/20], Loss: 0.0028
Epoch [17/20], Loss: 0.0037
Epoch [18/20], Loss: 0.0012
Epoch [19/20], Loss: 0.0018
Epoch [20/20], Loss: 0.0029
✅ Test Accuracy: 1.0000


['scaler.pkl']

In [4]:
# app.py

from flask import Flask, request, jsonify
import torch
import torch.nn as nn
import numpy as np
import joblib

app = Flask(__name__)

# Load model and scaler
class BotDetector(nn.Module):
    def __init__(self, input_dim):
        super(BotDetector, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.model(x)

model = BotDetector(input_dim=4)
model.load_state_dict(torch.load('bot_model.pth'))
model.eval()
scaler = joblib.load('scaler.pkl')

@app.route('/predict', methods=['POST'])
def predict():
    data = request.get_json()

    try:
        input_features = np.array([
            data['request_rate'],
            data['user_agent_score'],
            data['time_on_page'],
            data['navigation_depth']
        ]).reshape(1, -1)

        input_scaled = scaler.transform(input_features)
        input_tensor = torch.tensor(input_scaled, dtype=torch.float32)

        with torch.no_grad():
            prediction = model(input_tensor)
            is_bot = int(prediction.item() > 0.5)

        return jsonify({'is_bot': is_bot, 'confidence': float(prediction.item())})

    except Exception as e:
        return jsonify({'error': str(e)})

if __name__ == '__main__':
    app.run(port=5000)


 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
